In [1]:
import pandas as pd
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix

# --- Load your full merged dataset ---
all_df = pd.read_csv(r"C:\Users\misog\SCHOOL\Summer project\ML-football-odds\dataset\all_seasons.csv")  # adjust path/filename

# --- Chronological season split ---
train_seasons = ["21-22", "22-23", "23-24", "24-25"]
test_season = "25-26"

train_df = all_df[all_df["Season"].isin(train_seasons)].copy()
test_df = all_df[all_df["Season"] == test_season].copy()

print(f"Train shape: {train_df.shape}")
print(f"Test shape: {test_df.shape}")

# --- Columns to drop from features (leakage / non-numeric identifiers) ---
drop_cols = ["Date", "Time", "HomeTeam", "AwayTeam", "FTHG", "FTAG", "FTR", "Season"]

X_train = train_df.drop(columns=drop_cols)
y_train = train_df["FTR"]

X_test = test_df.drop(columns=drop_cols)
y_test = test_df["FTR"]

# --- Handle any missing values (early-season rolling stats will have NaNs) ---
print(f"NaNs in X_train: {X_train.isna().sum().sum()}")
print(f"NaNs in X_test: {X_test.isna().sum().sum()}")

X_train = X_train.fillna(0)
X_test = X_test.fillna(0)

# --- Scale features (crucial for KNN — distance-based algorithm) ---
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)  # transform only, never fit, on test data

# --- Train ---
knn = KNeighborsClassifier(
    n_neighbors=15,
    weights="distance",  # closer matches count more than distant ones
    n_jobs=-1
)
knn.fit(X_train_scaled, y_train)

# --- Predict & evaluate ---
y_pred = knn.predict(X_test_scaled)

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("Confusion Matrix (rows=actual, cols=predicted, order = classes below):")
print(knn.classes_)
print(confusion_matrix(y_test, y_pred, labels=knn.classes_))

Train shape: (1439, 148)
Test shape: (360, 148)
NaNs in X_train: 0
NaNs in X_test: 0

Classification Report:
              precision    recall  f1-score   support

           A       0.39      0.39      0.39       109
           D       0.16      0.05      0.08        99
           H       0.50      0.72      0.59       152

    accuracy                           0.44       360
   macro avg       0.35      0.39      0.35       360
weighted avg       0.37      0.44      0.39       360

Confusion Matrix (rows=actual, cols=predicted, order = classes below):
['A' 'D' 'H']
[[ 43  11  55]
 [ 41   5  53]
 [ 26  16 110]]
